In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpilerové průchody s podporou AI
Transpilerové průchody s podporou AI jsou průchody, které fungují jako přímá náhrada „tradičních" průchodů Qiskitu pro některé transpilační úlohy. Často dosahují lepších výsledků než stávající heuristické algoritmy (například nižší hloubku a počet hradel CNOT), a zároveň jsou výrazně rychlejší než optimalizační algoritmy jako řešiče splnitelnosti booleovských formulí. Transpilerové průchody s podporou AI lze spustit ve tvém lokálním prostředí nebo v cloudu prostřednictvím Qiskit Transpiler Service, pokud jsi součástí plánu IBM Quantum&reg; Premium Plan, Flex Plan nebo On-Prem (přes IBM Quantum Platform API) Plan.

> **Note:** Transpilerové průchody s podporou AI jsou ve stavu beta verze a mohou se měnit.
>     Pokud máš zpětnou vazbu nebo chceš kontaktovat vývojářský tým, použij tento [kanál v Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Aktuálně jsou k dispozici následující průchody:

**Průchody směrování**
 - `AIRouting`: Výběr rozmístění a směrování Circuit

**Průchody syntézy Circuit**
 - `AICliffordSynthesis`: Syntéza Cliffordových Circuit
 - `AILinearFunctionSynthesis`: Syntéza Circuit lineárních funkcí
 - `AIPermutationSynthesis`: Syntéza permutačních Circuit
 - `AIPauliNetworkSynthesis`: Syntéza Circuit Pauliho sítě

Chceš-li používat transpilerové průchody s podporou AI, nejprve [nainstaluj balíček `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). Více informací o dostupných možnostech najdeš v [dokumentaci API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler).

## Spuštění transpilerových průchodů s podporou AI lokálně nebo v cloudu
Chceš-li zdarma používat transpilerové průchody s podporou AI ve svém lokálním prostředí, nainstaluj `qiskit-ibm-transpiler` s některými dalšími závislostmi takto:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Bez těchto dalších závislostí běží transpilerové průchody s podporou AI v cloudu prostřednictvím Qiskit Transpiler Service (dostupného pouze pro uživatele plánů IBM Quantum Premium Plan, Flex Plan nebo On-Prem (přes IBM Quantum Platform API) Plan). Po instalaci dalších závislostí je výchozím režimem spuštění transpilerových průchodů s podporou AI tvůj lokální počítač.

## Průchod směrování AI
Průchod `AIRouting` funguje zároveň jako fáze rozmístění i fáze směrování. Lze ho použít v rámci `PassManager` takto:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Zde `backend` určuje, pro jakou mapu propojení se má provést směrování, `optimization_level` (1, 2 nebo 3) určuje výpočetní úsilí vynaložené v průběhu procesu (vyšší hodnota obvykle dává lepší výsledky, ale trvá déle) a `layout_mode` specifikuje způsob zpracování výběru rozmístění.
`layout_mode` zahrnuje následující možnosti:

- `keep`: Tato možnost respektuje rozmístění nastavené předchozími průchody Transpileru (nebo použije triviální rozmístění, pokud nebylo nastaveno). Obvykle se používá pouze tehdy, kdy Circuit musí běžet na konkrétních Qubitech zařízení. Často přináší horší výsledky, protože má méně prostoru pro optimalizaci.
- `improve`: Tato možnost používá rozmístění nastavené předchozími průchody Transpileru jako výchozí bod. Je užitečná, pokud máš dobrý počáteční odhad rozmístění; například pro Circuit sestavené tak, aby přibližně odpovídaly mapě propojení zařízení. Je také užitečná, pokud chceš vyzkoušet jiné specifické průchody rozmístění v kombinaci s průchodem `AIRouting`.
- `optimize`: Toto je výchozí režim. Funguje nejlépe pro obecné Circuit, u nichž nemusíš mít dobrý odhad rozmístění. Tento režim ignoruje předchozí výběry rozmístění.
- `local_mode`: Tento příznak určuje, kde průchod `AIRouting` běží. Pokud je `False`, `AIRouting` běží vzdáleně prostřednictvím Qiskit Transpiler Service. Pokud je `True`, balíček se pokusí spustit průchod ve tvém lokálním prostředí s možností přechodu do cloudového režimu, pokud potřebné závislosti nejsou nalezeny.

## Průchody syntézy Circuit s podporou AI
Průchody syntézy Circuit s podporou AI ti umožňují optimalizovat části různých typů Circuit ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauliho síť) jejich opětovnou syntézou. Typický způsob použití průchodu syntézy je následující:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Syntéza respektuje mapu propojení zařízení: lze ji bezpečně spustit po jiných průchodech směrování, aniž by narušila Circuit, takže celkový Circuit bude stále respektovat omezení zařízení. Ve výchozím nastavení syntéza nahradí původní pod-Circuit pouze tehdy, pokud syntetizovaný pod-Circuit vylepší originál (aktuálně se kontroluje pouze počet hradel CNOT), ale toto chování lze přepsat tak, aby Circuit vždy nahrazoval, nastavením `replace_only_if_better=False`.

Následující průchody syntézy jsou dostupné z `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Syntéza pro [Cliffordovy](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) Circuit (bloky hradel `H`, `S` a `CX`). Aktuálně až do devítiqubitových bloků.
- *AILinearFunctionSynthesis*: Syntéza pro Circuit [lineárních funkcí](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) (bloky hradel `CX` a `SWAP`). Aktuálně až do devítiqubitových bloků.
- *AIPermutationSynthesis*: Syntéza pro [permutační](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) Circuit (bloky hradel `SWAP`). Aktuálně dostupná pro bloky 65, 33 a 27 Qubitů.
- *AIPauliNetworkSynthesis*: Syntéza pro Circuit Pauliho sítě (bloky hradel `H`, `S`, `SX`, `CX`, `RX`, `RY` a `RZ`). Aktuálně až do šestiqubitových bloků.

Postupně plánujeme zvyšovat velikost podporovaných bloků.

Všechny průchody využívají fond vláken pro paralelní odesílání více požadavků. Ve výchozím nastavení je maximální počet vláken roven počtu jader plus čtyři (výchozí hodnoty pro objekt Pythonu `ThreadPoolExecutor`). Svou vlastní hodnotu však můžeš nastavit pomocí argumentu `max_threads` při vytváření instance průchodu. Například následující řádek vytváří instanci průchodu `AILinearFunctionSynthesis`, která mu umožňuje používat maximálně 20 vláken.